## Download Dataset from GitHub

We will clone the `Plants-have-cancer` repository, extract the `data` folder, and then remove the cloned repository.

In [1]:
# Clone the specific branch of the repository
!git clone --branch q0q https://github.com/IssacAnand/Plants-have-cancer.git

# Create the data directory if it doesn't exist in the current path
!mkdir -p data

# Move the 'data' folder from the cloned repository to the current directory
!mv Plants-have-cancer/backend/data/* data/

# Clean up: Remove the cloned repository
!rm -rf Plants-have-cancer/

print("Dataset downloaded and moved to './data' folder.")
print("Contents of './data/':")
!ls -R data/

Streaming output truncated to the last 5000 lines.
140.jpg  216.jpg  291.jpg  63.jpg   google_0070.jpg  google_0432.jpg
141.jpg  217.jpg  292.jpg  64.jpg   google_0072.jpg  google_0437.jpg
142.jpg  219.jpg  294.jpg  65.jpg   google_0073.jpg  google_0446.jpg
143.jpg  21.jpg   295.jpg  66.jpg   google_0075.jpg  google_0463.jpg
144.jpg  220.jpg  296.jpg  67.jpg   google_0076.jpg  google_0466.jpg
145.jpg  221.jpg  297.jpg  68.jpg   google_0078.jpg  google_0474.jpg
146.jpg  222.jpg  298.jpg  69.jpg   google_0079.jpg  google_0479.jpg
147.jpg  223.jpg  299.jpg  6.jpg    google_0081.jpg  google_0480.jpg
149.jpg  224.jpg  29.jpg   70.jpg   google_0092.jpg  google_0481.jpg
14.jpg	 225.jpg  2.jpg    71.jpg   google_0094.jpg  google_0493.jpg
150.jpg  226.jpg  300.jpg  72.jpg   google_0110.jpg  google_0494.jpg
151.jpg  227.jpg  301.jpg  74.jpg   google_0112.jpg  google_0497.jpg
152.jpg  228.jpg  302.jpg  75.jpg   google_0118.jpg  google_0498.jpg
153.jpg  229.jpg  303.jpg  77.jpg   google_0120.jpg  

# PlantWild Multimodal Pipeline

End-to-end pipeline for plant disease classification using MobileViTv2 (image) + agriculture-BERT (text), with gradient-based heatmap generation.

**Pipeline order:**
1. Dataset
2. MobileViT Image Encoder
3. BERT Text Encoder
4. Multimodal MLP
5. Generate Heatmap Data
6. Train Heatmap Generator

---
# Dataset

_Shared dataset classes used across all pipeline stages._

In [4]:
from pathlib import Path
import json
from PIL import Image

import torch
from torch.utils.data import Dataset

class PlantWildDataset(Dataset):
    """
    Args:
        images_dir (str) : Path to plantwild/ folder.
        transform        : torchvision transform pipeline.
        label_map (dict) : {class_name: int}. Auto-built from folders if None.
        split     (str)  : "train" | "test" | "all". Default "all".
        test_size (float): Fraction reserved for test. Default 0.2.
        seed      (int)  : Random seed for reproducible splits. Default 42.
    """

    def __init__(self, images_dir: str, transform=None, label_map: dict = None,
                 split: str = "all", test_size: float = 0.2, seed: int = 42):
        self.images_dir = Path(images_dir)
        self.transform  = transform

        class_dirs     = sorted([d for d in self.images_dir.iterdir() if d.is_dir()])
        self.label_map = label_map or {d.name: i for i, d in enumerate(class_dirs)}
        self.classes   = list(self.label_map.keys())

        all_samples = [
            (img_path, self.label_map[cls_dir.name])
            for cls_dir in class_dirs
            if cls_dir.name in self.label_map
            for img_path in sorted(cls_dir.glob("*.jpg"),
                                   key=lambda p: int(p.stem) if p.stem.isdigit() else float("inf"))
        ]

        if split == "all":
            self.samples = all_samples
        else:
            generator = torch.Generator().manual_seed(seed)
            indices   = torch.randperm(len(all_samples), generator=generator).tolist()
            n_test    = int(len(all_samples) * test_size)
            n_train   = len(all_samples) - n_test
            train_idx = indices[:n_train]
            test_idx  = indices[n_train:]
            self.samples = [all_samples[i] for i in (train_idx if split == "train" else test_idx)]

        print(f"PlantWildDataset | split={split} | "
              f"{len(self.classes)} classes | {len(self.samples)} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    def save_label_map(self, path: str):
        with open(path, "w") as f:
            json.dump(self.label_map, f, indent=2)
        print(f"Label map saved → {path}")

    def get_class_counts(self) -> torch.Tensor:
        """Returns a tensor of per-class sample counts for weighted loss."""
        counts = torch.zeros(len(self.classes))
        for _, label in self.samples:
            counts[label] += 1
        return counts

class PlantWildTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.samples=df[["description", "label"]].values.tolist()
        self.tokenizer=tokenizer
        self.max_length=max_length

        print(df.head())

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        text, label = self.samples[index]
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

---
# MobileViT Image Encoder

_Fine-tune MobileViTv2 on plant images and extract image embeddings._

In [5]:
!pip install evaluate

import os
import json
from pathlib import Path
from datetime import datetime
import numpy as np
from tqdm import tqdm

import torch
import torchvision
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import GradScaler, autocast
from torchvision import transforms
from PIL import Image
import timm
import evaluate



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00


## Config

In [7]:
MODEL_NAME  = "mobilevitv2_150"
IMAGES_DIR  = "./data/images/plantwild"
IMG_SIZE    = 320
BATCH_SIZE  = 16
EPOCHS      = 50
BACKBONE_LR = 1e-5
HEAD_LR     = 1e-3
SAVE_DIR    = f"./checkpoints"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXCLUDED_SUFFIX = "leaf"

torch.cuda.manual_seed(42)
torch.manual_seed(42)

metric_f1  = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

## Model

In [8]:
class MobileViT(nn.Module):
    """
    Full MobileViT fine-tune with all layers trainable.
    """

    def __init__(self, num_classes: int,
                 model_name: str, dropout: float = 0.2):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        self.embed_dim = self.backbone.num_features

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, num_classes),
        )

        for param in self.backbone.parameters():
            param.requires_grad = True

        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Model           : {model_name}")
        print(f"Total params    : {total / 1e6:.2f}M")
        print(f"Trainable params: {trainable / 1e6:.2f}M  "
              f"({100 * trainable / total:.1f}% unfrozen)")

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(images))

    def get_encoder(self):
        return self.backbone

    def get_param_groups(self, backbone_lr: float, head_lr: float):
        """
        Separate LRs for backbone and head.
        Backbone gets a lower LR to preserve pretrained weights.
        """
        return [
            {"params": self.backbone.parameters(), "lr": backbone_lr},
            {"params": self.head.parameters(),     "lr": head_lr},
        ]

## Transforms

In [9]:
def get_transforms(img_size: int = IMG_SIZE, train: bool = True):
    """
    Data augmentation and normalization similar to pre-trained ImageNet configs.
    """
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def build_filtered_label_map(images_dir: str):
    """Build a label map that excludes class folders ending with 'leaf'."""
    class_dirs = sorted([d for d in Path(images_dir).iterdir() if d.is_dir()])
    excluded = [d.name for d in class_dirs if d.name.endswith(EXCLUDED_SUFFIX)]
    kept = [d.name for d in class_dirs if not d.name.endswith(EXCLUDED_SUFFIX)]

    if not kept:
        raise ValueError(f"No classes remain after excluding labels ending with '{EXCLUDED_SUFFIX}'.")

    print(f"Excluding {len(excluded)} classes ending with '{EXCLUDED_SUFFIX}':")
    print(", ".join(excluded))
    print(f"Keeping {len(kept)} classes for MobileViT training/testing.")

    return {class_name: idx for idx, class_name in enumerate(kept)}

## Training

In [10]:
def train(model, train_loader, test_loader, device, save_dir, class_weights=None):

    os.makedirs(save_dir, exist_ok=True)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None,
        label_smoothing=0.1,
    )

    optimizer = torch.optim.AdamW(
        model.get_param_groups(BACKBONE_LR, HEAD_LR), weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2
    )

    scaler   = GradScaler("cuda")
    best_acc = 0.0
    best_f1 = 0.0

    # Training loop ---------------------------------------------------------------------------
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            with autocast("cuda"):  # auto switch between fp32 and fp16 for sensitive / non-sensitive computations
                loss = criterion(model(images), labels)

            scaler.scale(loss).backward()   # scale the loss to prevent underflow in fp16
            scaler.unscale_(optimizer)      # scale back down
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # cap gradients to prevent gradient explosion from bad batch
            scaler.step(optimizer)      # check if any gradients are inf/nan, and skip optimizer step if yes
            scaler.update()             # update the scale factor for next iteration

            train_loss += loss.item()

        scheduler.step(epoch)

        # Test loop -------------------------------------------------------------------------------
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in tqdm(test_loader, desc="Testing", leave=True):
                images, labels = images.to(device), labels.to(device)
                with autocast("cuda"):
                    preds = model(images).argmax(1)
                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        f1  = metric_f1.compute(predictions=all_preds, references=all_labels, average="macro")
        acc = metric_acc.compute(predictions=all_preds, references=all_labels)
        macro_f1 = f1["f1"] * 100
        accuracy = acc["accuracy"] * 100

        print(f"Epoch {epoch:>3}/{EPOCHS}  "
              f"Loss: {train_loss / len(train_loader):.4f}  "
              f"Test Acc: {accuracy:.2f}%  "
              f"Macro F1: {macro_f1:.2f}%")

        if accuracy > best_acc:
            best_acc = accuracy
            best_f1 = macro_f1
            torch.save(
                model.get_encoder().state_dict(),
                os.path.join(save_dir, "best_image_encoder.pt")
            )
            print(f"  ✓ Best encoder saved (Acc: {best_acc:.2f}%  Macro F1: {best_f1:.2f}%)")

    print(f"\nFinetuning complete — best Test Acc: {best_acc:.2f}%  Macro F1: {best_f1:.2f}%")
    return model

## Embedding Extraction

In [11]:
def extract_embeddings(encoder, dataloader, device):
    encoder.eval()
    all_embeddings, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting", leave=True):
            images = images.to(device)
            with autocast("cuda"):
                embeddings = encoder(images)
            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels)
    return torch.cat(all_embeddings, dim=0), torch.cat(all_labels, dim=0)


def save_embeddings(encoder, train_ds, test_loader, save_dir, device):
    # use test transforms — no augmentation for embedding extraction
    train_ds_eval = PlantWildDataset(
        IMAGES_DIR,
        transform=get_transforms(IMG_SIZE, train=False),
        split="train",
        label_map=train_ds.label_map,
    )
    train_loader_eval = DataLoader(
        train_ds_eval, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True, persistent_workers=True,
    )

    print("Extracting train embeddings...")
    train_emb, train_lbl = extract_embeddings(encoder, train_loader_eval, device)
    print("Extracting test embeddings...")
    test_emb, test_lbl   = extract_embeddings(encoder, test_loader, device)

    print(f"Train embeddings : {train_emb.shape}")   # (14832, 768)
    print(f"Test embeddings  : {test_emb.shape}")    # (3708, 768)

    save_path = os.path.join(save_dir, "image_embeddings.pt")
    torch.save({
        "train_embeddings": train_emb,
        "train_labels":     train_lbl,
        "test_embeddings":  test_emb,
        "test_labels":      test_lbl,
    }, save_path)
    print(f"Embeddings saved → {save_path}")

## Main

In [12]:

if __name__ == "__main__":
    print(f"Device : {DEVICE}")
    print(f"Model  : {MODEL_NAME}")

    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    label_map = build_filtered_label_map(IMAGES_DIR)

    # Load datasets
    train_ds = PlantWildDataset(IMAGES_DIR,
                                transform=get_transforms(IMG_SIZE, train=True),
                                split="train",
                                label_map=label_map)
    test_ds  = PlantWildDataset(IMAGES_DIR,
                                transform=get_transforms(IMG_SIZE, train=False),
                                split="test", label_map=label_map)

    train_ds.save_label_map("./data/label_map.json")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True,
                              persistent_workers=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True,
                              persistent_workers=True)

    # Train model
    model = MobileViT(
        num_classes=len(train_ds.classes),
        model_name=MODEL_NAME,
        dropout=0.2,
    )

    class_counts  = train_ds.get_class_counts()
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum()
    print(f"Class weights computed for {len(class_weights)} classes")

    model = train(model, train_loader, test_loader, DEVICE, SAVE_DIR, class_weights=class_weights)

    # Extract embeddings
    encoder = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
    encoder.load_state_dict(
        torch.load(os.path.join(SAVE_DIR, "best_image_encoder.pt"),
                   map_location=DEVICE)
    )
    encoder = encoder.to(DEVICE)
    save_embeddings(encoder, train_ds, test_loader, SAVE_DIR, DEVICE)

Device : cuda
Model  : mobilevitv2_150
Excluding 30 classes ending with 'leaf':
apple leaf, banana leaf, basil leaf, bean leaf, bell pepper leaf, blueberry leaf, broccoli leaf, cabbage leaf, cauliflower leaf, celery leaf, cherry leaf, coffee leaf, corn leaf, cucumber leaf, eggplant leaf, garlic leaf, ginger leaf, grape leaf, lettuce leaf, maple leaf, peach leaf, plum leaf, potato leaf, raspberry leaf, rice leaf, soybean leaf, squash leaf, strawberry leaf, tobacco leaf, tomato leaf
Keeping 59 classes for MobileViT training/testing.
PlantWildDataset | split=train | 59 classes | 8560 images
PlantWildDataset | split=test | 59 classes | 2140 images
Label map saved → ./data/label_map.json


model.safetensors:   0%|          | 0.00/42.5M [00:00<?, ?B/s]

Model           : mobilevitv2_150
Total params    : 9.87M
Trainable params: 9.87M  (100.0% unfrozen)
Class weights computed for 59 classes


Testing: 100%|██████████| 134/134 [00:21<00:00,  6.15it/s]


Epoch   1/50  Loss: 3.3569  Test Acc: 49.30%  Macro F1: 45.50%
  ✓ Best encoder saved (Acc: 49.30%  Macro F1: 45.50%)


...

Epoch  50/50  Loss: 1.3935  Test Acc: 66.54%  Macro F1: 64.23%

Finetuning complete — best Test Acc: 66.68%  Macro F1: 64.57%
PlantWildDataset | split=train | 59 classes | 8560 images
Extracting train embeddings...


Extracting: 100%|██████████| 535/535 [00:38<00:00, 13.84it/s]


Extracting test embeddings...


Extracting: 100%|██████████| 134/134 [00:18<00:00,  7.19it/s]

Train embeddings : torch.Size([8560, 768])
Test embeddings  : torch.Size([2140, 768])
Embeddings saved → ./checkpoints/image_embeddings.pt


---
# BERT Text Encoder

_Fine-tune agriculture-BERT on disease descriptions and extract text embeddings._

In [13]:
import os
import re
import json
import sys
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate



## Config

In [14]:
MODEL_NAME     = "recobo/agriculture-bert-uncased"
DATASET_PATH   = "./data/text/plantwild.json"
LABEL_MAP_PATH = "./data/label_map.json"
SAVE_DIR       = "./checkpoints"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

MAX_LENGTH     = 128
BATCH_SIZE     = 8
NUM_EPOCHS     = 10
LEARNING_RATE  = 2e-5
WEIGHT_DECAY   = 0.01
EXCLUDED_SUFFIX = "leaf"

## Data Preprocessing

In [15]:
def remove_disease_name(row):
    disease     = str(row["disease"]).strip()
    description = str(row["description"]).strip()

    if not disease or not description:
        return description

    cleaned = re.sub(re.escape(disease), "", description, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" ,.-:;")
    return cleaned


def preprocess_text_data(path, label_map):
    df = pd.read_json(path)

    # reshape wide -> long
    df_long = df.melt(var_name="disease", value_name="description")

    # clean
    df_long = df_long.dropna()
    df_long["description"] = df_long["description"].astype(str).str.strip()
    df_long = df_long[df_long["description"] != ""]

    # remove disease name from descriptions
    df_long["description"] = df_long.apply(remove_disease_name, axis=1)

    # create numeric label from existing map
    df_long["label"] = df_long["disease"].map(label_map)
    df_long = df_long.dropna(subset=["label"])
    df_long["label"] = df_long["label"].astype(int)

    return df_long


def load_filtered_label_map():
    """Load label_map.json and exclude labels ending with 'leaf'."""
    with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
        raw_label_map = json.load(f)

    ordered_labels = sorted(raw_label_map.items(), key=lambda item: item[1])
    excluded = [name for name, _ in ordered_labels if name.endswith(EXCLUDED_SUFFIX)]
    kept = [name for name, _ in ordered_labels if not name.endswith(EXCLUDED_SUFFIX)]

    if not kept:
        raise ValueError(f"No classes remain after excluding labels ending with '{EXCLUDED_SUFFIX}'.")

    print(f"Excluding {len(excluded)} classes ending with '{EXCLUDED_SUFFIX}':")
    print(", ".join(excluded))
    print(f"Keeping {len(kept)} classes for BERT training/testing.")

    return {name: idx for idx, name in enumerate(kept)}


def build_datasets(tokenizer, label_map):

    preprocessed_df = preprocess_text_data(DATASET_PATH, label_map)

    train_df, test_df = train_test_split(
        preprocessed_df,
        test_size=0.2,
        stratify=preprocessed_df["label"],
        random_state=42,
    )

    train_dataset = PlantWildTextDataset(train_df, tokenizer, max_length=MAX_LENGTH)
    test_dataset  = PlantWildTextDataset(test_df,  tokenizer, max_length=MAX_LENGTH)

    print(f"Train samples: {len(train_df)}  Test samples: {len(test_df)}")

    return train_dataset, test_dataset, train_df, test_df

## Metrics

In [16]:
metric_f1  = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    f1  = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    acc = metric_acc.compute(predictions=predictions, references=labels)
    return {**f1, **acc}

## Training

In [17]:
def train(model, train_dataset, test_dataset):

    model_slug = MODEL_NAME.split("/")[-1]

    args = TrainingArguments(
        os.path.join(SAVE_DIR, "trainer_tmp"),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        push_to_hub=False,
        fp16=True,
    )

    trainer = Trainer(
        model,
        args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    print(metrics)

    os.makedirs(SAVE_DIR, exist_ok=True)
    trainer.save_model(os.path.join(SAVE_DIR, "best_text_encoder.pt"))

    return trainer

## Evaluation

In [18]:
def evaluate_predictions(trainer, test_dataset, label_map, test_df):
    """Run predictions and print F1 / accuracy. Also logs top misclassifications."""
    pred_output = trainer.predict(test_dataset)
    logits      = pred_output.predictions
    y_pred      = np.argmax(logits, axis=-1)
    y_true      = pred_output.label_ids

    id_to_label = {v: k for k, v in label_map.items()}
    pred_names  = [id_to_label[int(x)] for x in y_pred]
    true_names  = [id_to_label[int(x)] for x in y_true]

    f1  = metric_f1.compute(predictions=y_pred, references=y_true, average="macro")
    acc = metric_acc.compute(predictions=y_pred, references=y_true)

    print("F1:",       f1)
    print("Accuracy:", acc)

    # Misclassification analysis ------------------------------------------------------------------
    raw_df               = pd.read_json(DATASET_PATH)
    label_descriptions_df = pd.DataFrame([
        {
            "Disease Name":       disease,
            "Sample Description": raw_df[disease].dropna().iloc[0] if not raw_df[disease].dropna().empty else "",
        }
        for disease in raw_df.columns
    ])

    results_df = pd.DataFrame({
        "True Label":      true_names,
        "Predicted Label": pred_names,
        "Description":     test_df["description"].reset_index(drop=True),
    })

    misclassified_df = results_df[results_df["True Label"] != results_df["Predicted Label"]]

    for side in [("True Label", "True Label Description"), ("Predicted Label", "Predicted Label Description")]:
        col, rename = side
        misclassified_df = pd.merge(
            misclassified_df,
            label_descriptions_df[["Disease Name", "Sample Description"]],
            left_on=col, right_on="Disease Name",
            suffixes=("", f"_{col}"),
        )
        misclassified_df = misclassified_df.drop(columns=["Disease Name"])
        misclassified_df = misclassified_df.rename(columns={"Sample Description": rename})

    misclassification_counts      = misclassified_df.groupby(["True Label", "Predicted Label"]).size().reset_index(name="Count")
    top_misclassifications_summary = misclassification_counts.sort_values(by="Count", ascending=False).head(10)

    print("\nTop 10 misclassified pairs:")
    print(top_misclassifications_summary.to_string(index=False))

    print("\nTop 10 individual misclassified samples:")
    pd.set_option("display.max_colwidth", None)
    print(misclassified_df.head(10).to_string(index=False))
    pd.reset_option("display.max_colwidth")

## Embedding Extraction

In [19]:
class FeatureExtractorModel(torch.nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.bert = original_model.bert

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] token embedding (index 0) used as sentence representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return cls_embedding


def extract_features(dataset, model, device, batch_size=16):
    """Extract [CLS] token embeddings from BERT for all samples in a dataset."""
    feature_model = FeatureExtractorModel(model)
    feature_model.to(device)
    feature_model.eval()

    all_features = []
    all_labels   = []
    dataloader   = DataLoader(dataset, batch_size=batch_size, shuffle=False)  # shuffle=False is critical

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"]
            features       = feature_model(input_ids=input_ids, attention_mask=attention_mask)
            all_features.append(features.cpu())
            all_labels.append(labels)

    return torch.cat(all_features, dim=0), torch.cat(all_labels, dim=0)


def save_features(train_dataset, test_dataset, model, tokenizer):
    """Load best model, extract features for train/test, and save to disk."""
    model     = AutoModelForSequenceClassification.from_pretrained(os.path.join(SAVE_DIR, "best_text_encoder.pt"))
    tokenizer = AutoTokenizer.from_pretrained(os.path.join(SAVE_DIR, "best_text_encoder.pt"))
    model.to(DEVICE)
    model.eval()

    train_features, train_labels = extract_features(train_dataset, model, DEVICE)
    test_features,  test_labels  = extract_features(test_dataset,  model, DEVICE)

    print(f"Train text features: {train_features.shape}")
    print(f"Test  text features: {test_features.shape}")

    os.makedirs(SAVE_DIR, exist_ok=True)

    torch.save(
        {
            "train_features": train_features,
            "train_labels":   train_labels,
            "test_features":  test_features,
            "test_labels":    test_labels,
        },
        os.path.join(SAVE_DIR, "text_embeddings.pt"),
    )

    print(f"\n✓ Features saved to {os.path.join(SAVE_DIR, 'text_embeddings.pt')}")

## Main

In [20]:
def main():
    print(f"PyTorch {torch.__version__}")
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    label_map = load_filtered_label_map()

    # Tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(label_map)
    )

    print(f"Output labels:  {model.config.num_labels}")
    print(f"Hidden size:    {model.config.hidden_size}")

    train_dataset, test_dataset, train_df, test_df = build_datasets(tokenizer, label_map)

    # Train
    print(f"\nFine-tuning {MODEL_NAME} for {NUM_EPOCHS} epochs...\n")
    trainer = train(model, train_dataset, test_dataset)

    # Save tokenizer alongside model
    tokenizer.save_pretrained(os.path.join(SAVE_DIR, "best_text_encoder.pt"))

    # Evaluate
    print("\nRunning final evaluation...")
    evaluate_predictions(trainer, test_dataset, label_map, test_df)

    # Extract and save features
    print("\nExtracting BERT features...")
    save_features(train_dataset, test_dataset, model, tokenizer)


if __name__ == "__main__":
    main()

PyTorch 2.10.0+cu128
Device: cuda
GPU: NVIDIA L4
Excluding 0 classes ending with 'leaf':

Keeping 59 classes for BERT training/testing.


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

{'eval_loss': 0.6873918175697327, 'eval_f1': 0.7801810898317031, 'eval_accuracy': 0.7779661016949152, 'eval_runtime': 1.3305, 'eval_samples_per_second': 443.427, 'eval_steps_per_second': 55.616, 'epoch': 10.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Running final evaluation...
F1: {'f1': 0.7801810898317031}
Accuracy: {'accuracy': 0.7779661016949152}

Top 10 misclassified pairs:
                      True Label              Predicted Label  Count
             potato early blight          tomato early blight      5
           bell pepper leaf spot             ginger leaf spot      4
                ginger leaf spot        bell pepper leaf spot      4
             celery early blight          potato early blight      4
             tomato early blight          potato early blight      4
         cucumber powdery mildew        squash powdery mildew      4
           banana panama disease      cucumber bacterial wilt      4
         cucumber bacterial wilt        banana panama disease      3
                       corn rust                  garlic rust      3
cauliflower alternaria leaf spot cabbage alternaria leaf spot      3

Top 10 individual misclassified samples:
                  True Label                  Predicted Label      

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Train text features: torch.Size([2360, 768])
Test  text features: torch.Size([590, 768])

✓ Features saved to ./checkpoints/text_embeddings.pt


---
# Multimodal MLP

_Fuse image and text embeddings and train the classifier._

In [31]:
import json
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm

## Config

In [32]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS      = 50
LR          = 1e-3
BATCH_SIZE  = 32
DROPOUT     = 0.3

IMAGE_EMB_PATH = "./checkpoints/image_embeddings.pt"
TEXT_EMB_PATH  = "./checkpoints/text_embeddings.pt"
# MLP_SAVE       = "./checkpoints/best_multimodal_mlp.pt"
MLP_SAVE_DIR = "./checkpoints"
LABEL_MAP_PATH = Path("./data/label_map.json")
EXCLUDED_SUFFIX = "leaf"

ABLATION_MODE = "multimodal"

if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
torch.manual_seed(42)

## Model

In [33]:
class FlexibleMLP(nn.Module):
    """
    Classifier for multimodal, BERT-only, or ViT-only ablation.
    """

    def __init__(
        self,
        image_dim=768,
        text_dim=768,
        num_classes=None,
        mode="multimodal",
        dropout=DROPOUT,
    ):
        super().__init__()

        if num_classes is None:
            raise ValueError("num_classes must be provided.")

        if mode not in {"multimodal", "ViT", "BERT"}:
            raise ValueError("mode must be one of {'multimodal', 'ViT', 'BERT'}")

        self.mode = mode

        if mode == "multimodal":
            input_dim = image_dim + text_dim
        elif mode == "ViT":
            input_dim = image_dim
        elif mode == "BERT":
            input_dim = text_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, num_classes),
        )

    def forward(self, image_emb=None, text_emb=None):
        if self.mode == "multimodal":
            if image_emb is None or text_emb is None:
                raise ValueError("multimodal mode requires both image_emb and text_emb")
            x = torch.cat([image_emb, text_emb], dim=1)

        elif self.mode == "ViT":
            if image_emb is None:
                raise ValueError("ViT mode requires image_emb")
            x = image_emb

        elif self.mode == "BERT":
            if text_emb is None:
                raise ValueError("BERT mode requires text_emb")
            x = text_emb

        x = x.float()
        return self.mlp(x)

## Data

In [34]:
def load_embeddings():
    """Load image and text embeddings."""
    image_data = torch.load(IMAGE_EMB_PATH)
    img_train_embeddings = image_data["train_embeddings"]
    img_train_labels     = image_data["train_labels"]
    img_test_embeddings  = image_data["test_embeddings"]
    img_test_labels      = image_data["test_labels"]

    text_data = torch.load(TEXT_EMB_PATH)
    txt_train_embeddings = text_data["train_features"]
    txt_train_labels     = text_data["train_labels"]
    txt_test_embeddings  = text_data["test_features"]
    txt_test_labels      = text_data["test_labels"]

    print(f"Image train embeddings shape:  {img_train_embeddings.shape}")
    print(f"Text  train embeddings shape:  {txt_train_embeddings.shape}")
    print(f"Image test  embeddings shape:  {img_test_embeddings.shape}")
    print(f"Text  test  embeddings shape:  {txt_test_embeddings.shape}")
    print(f"Unique image classes: {img_train_labels.unique().shape[0]}")
    print(f"Unique text  classes: {txt_train_labels.unique().shape[0]}")

    return (img_train_embeddings, img_train_labels,
            img_test_embeddings,  img_test_labels,
            txt_train_embeddings, txt_train_labels,
            txt_test_embeddings,  txt_test_labels)


def load_filtered_label_info():
    """Return the class IDs to keep after excluding labels that end with 'leaf'."""
    with LABEL_MAP_PATH.open(encoding="utf-8") as f:
        label_map = json.load(f)

    ordered_labels = sorted(label_map.items(), key=lambda item: item[1])
    excluded = [(name, idx) for name, idx in ordered_labels if name.endswith(EXCLUDED_SUFFIX)]
    kept = [(name, idx) for name, idx in ordered_labels if not name.endswith(EXCLUDED_SUFFIX)]

    if not kept:
        raise ValueError(f"No classes remain after excluding labels ending with '{EXCLUDED_SUFFIX}'.")

    old_to_new = {old_idx: new_idx for new_idx, (_, old_idx) in enumerate(kept)}

    print(f"Excluding {len(excluded)} classes ending with '{EXCLUDED_SUFFIX}':")
    print(", ".join(name for name, _ in excluded))
    print(f"Keeping {len(kept)} classes for training/testing.")

    return {
        "excluded": excluded,
        "kept": kept,
        "old_to_new": old_to_new,
        "num_classes": len(kept),
    }


def filter_and_remap_embeddings(embeddings, labels, old_to_new):
    """Drop excluded labels and remap the remaining labels to 0..N-1."""
    remap = torch.full((max(old_to_new) + 1,), -1, dtype=torch.long)
    for old_idx, new_idx in old_to_new.items():
        remap[old_idx] = new_idx

    mapped_labels = remap[labels.long()]
    keep_mask = mapped_labels >= 0

    return embeddings[keep_mask], mapped_labels[keep_mask]


def align_embeddings(img_emb, img_lbl, txt_emb, txt_lbl, num_classes):
    """
    Align image and text embeddings by class label.
    Keeps all image samples; samples text embeddings with replacement
    to match the image count per class.
    """
    aligned_img, aligned_txt, aligned_lbl = [], [], []

    for class_idx in range(num_classes):
        img_mask  = (img_lbl == class_idx)
        txt_mask  = (txt_lbl == class_idx)
        img_class = img_emb[img_mask]
        txt_class = txt_emb[txt_mask]

        if len(img_class) == 0 or len(txt_class) == 0:
            continue

        n = len(img_class)
        repeat_idx = torch.randint(0, len(txt_class), (n,))
        aligned_img.append(img_class)
        aligned_txt.append(txt_class[repeat_idx])
        aligned_lbl.append(torch.full((n,), class_idx, dtype=torch.long))

    if not aligned_img:
        raise ValueError("No aligned classes found after filtering/remapping embeddings.")

    return (torch.cat(aligned_img),
            torch.cat(aligned_txt),
            torch.cat(aligned_lbl))


def build_loaders(train_img, train_txt, train_lbl,
                  test_img,  test_txt,  test_lbl):
    train_img = train_img.float()
    train_txt = train_txt.float()
    test_img = test_img.float()
    test_txt = test_txt.float()

    train_dataset = TensorDataset(train_img, train_txt, train_lbl)
    test_dataset  = TensorDataset(test_img,  test_txt,  test_lbl)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, test_loader

## Training

In [36]:
def train(model, train_loader, test_loader, mode="multimodal"):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2
    )

    save_path = os.path.join(MLP_SAVE_DIR, f"best_mlp_{mode}.pt")
    os.makedirs(MLP_SAVE_DIR, exist_ok=True)

    best_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for img_emb, txt_emb, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=True):
            img_emb = img_emb.to(DEVICE).float()
            txt_emb = txt_emb.to(DEVICE).float()

            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            if mode == "multimodal":
                logits = model(image_emb=img_emb, text_emb=txt_emb)
            elif mode == "ViT":
                logits = model(image_emb=img_emb)
            elif mode == "BERT":
                logits = model(text_emb=txt_emb)
            else:
                raise ValueError(f"Unknown mode: {mode}")

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_correct += (logits.argmax(dim=1) == labels).sum().item()
            train_total += labels.size(0)

        scheduler.step()

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for img_emb, txt_emb, labels in tqdm(test_loader, desc="Testing", leave=True):
                img_emb = img_emb.to(DEVICE)
                txt_emb = txt_emb.to(DEVICE)
                labels = labels.to(DEVICE)

                if mode == "multimodal":
                    logits = model(image_emb=img_emb, text_emb=txt_emb)
                elif mode == "ViT":
                    logits = model(image_emb=img_emb)
                else:
                    logits = model(text_emb=txt_emb)

                preds = logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        train_acc = 100.0 * train_correct / train_total
        test_acc = 100.0 * correct / total

        print(
            f"Epoch {epoch:>3}/{EPOCHS}  "
            f"Loss: {train_loss / len(train_loader):.4f}  "
            f"Train Acc: {train_acc:.2f}%  "
            f"Test Acc: {test_acc:.2f}%"
        )

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Best model saved to {save_path} ({best_acc:.2f}%)")

    print(f"\nDone — best Test Acc: {best_acc:.2f}%")
    return best_acc, save_path

## Evaluation

In [37]:
def evaluate_full(model, loader, mode="multimodal"):
    """Compute accuracy, macro precision, recall, and F1."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for img_emb, txt_emb, labels in tqdm(loader, desc="Evaluating", leave=True):
            img_emb = img_emb.to(DEVICE).float()
            txt_emb = txt_emb.to(DEVICE).float()
            labels = labels.to(DEVICE)

            if mode == "multimodal":
                logits = model(image_emb=img_emb, text_emb=txt_emb)
            elif mode == "ViT":
                logits = model(image_emb=img_emb)
            elif mode == "BERT":
                logits = model(text_emb=txt_emb)
            else:
                raise ValueError(f"Unknown mode: {mode}")

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc       = 100 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    recall    = recall_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    f1        = f1_score(all_labels, all_preds, average="macro", zero_division=0) * 100

    return acc, precision, recall, f1

## Main

In [38]:
def main():
    print(f"PyTorch {torch.__version__}")
    print(f"Device: {DEVICE}")
    print(f"Ablation mode: {ABLATION_MODE}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        torch.cuda.manual_seed(42)

    torch.manual_seed(42)

    label_info = load_filtered_label_info()

    # Load raw embeddings
    (img_train_emb, img_train_lbl,
     img_test_emb,  img_test_lbl,
     txt_train_emb, txt_train_lbl,
     txt_test_emb,  txt_test_lbl) = load_embeddings()

    # Filter/remap labels
    img_train_emb, img_train_lbl = filter_and_remap_embeddings(
        img_train_emb, img_train_lbl, label_info["old_to_new"]
    )
    img_test_emb, img_test_lbl = filter_and_remap_embeddings(
        img_test_emb, img_test_lbl, label_info["old_to_new"]
    )
    txt_train_emb, txt_train_lbl = filter_and_remap_embeddings(
        txt_train_emb, txt_train_lbl, label_info["old_to_new"]
    )
    txt_test_emb, txt_test_lbl = filter_and_remap_embeddings(
        txt_test_emb, txt_test_lbl, label_info["old_to_new"]
    )

    print("\nFiltered embeddings:")
    print(f"Image train embeddings shape:  {img_train_emb.shape}")
    print(f"Text  train embeddings shape:  {txt_train_emb.shape}")
    print(f"Image test  embeddings shape:  {img_test_emb.shape}")
    print(f"Text  test  embeddings shape:  {txt_test_emb.shape}")
    print(f"Unique filtered image classes: {img_train_lbl.unique().shape[0]}")
    print(f"Unique filtered text  classes: {txt_train_lbl.unique().shape[0]}")

    print("\nAligning embeddings...")
    train_img, train_txt, train_lbl = align_embeddings(
        img_train_emb, img_train_lbl,
        txt_train_emb, txt_train_lbl,
        label_info["num_classes"]
    )
    test_img, test_txt, test_lbl = align_embeddings(
        img_test_emb, img_test_lbl,
        txt_test_emb, txt_test_lbl,
        label_info["num_classes"]
    )

    print(f"Aligned train: img={train_img.shape}, txt={train_txt.shape}, lbl={train_lbl.shape}")
    print(f"Aligned test:  img={test_img.shape}, txt={test_txt.shape}, lbl={test_lbl.shape}")

    train_loader, test_loader = build_loaders(
        train_img, train_txt, train_lbl,
        test_img, test_txt, test_lbl
    )

    model = FlexibleMLP(
        image_dim=train_img.shape[1],
        text_dim=train_txt.shape[1],
        num_classes=label_info["num_classes"],
        mode=ABLATION_MODE,
        dropout=DROPOUT
    ).to(DEVICE)

    print(f"\nTraining for {EPOCHS} epochs on {DEVICE}...\n")
    best_acc, best_model_path = train(model, train_loader, test_loader, mode=ABLATION_MODE)

    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

    print("\nFinal evaluation on test set:")
    acc, precision, recall, f1 = evaluate_full(model, test_loader, mode=ABLATION_MODE)
    print(
        f"  Acc: {acc:.2f}%  "
        f"Precision: {precision:.2f}%  "
        f"Recall: {recall:.2f}%  "
        f"F1: {f1:.2f}%"
    )

# if __name__ == "__main__":
#     main()

In [39]:
for mode in ["multimodal", "ViT", "BERT"]:
    ABLATION_MODE = mode
    print(f"\n{'='*60}\nRunning mode: {mode}\n{'='*60}")
    main()


Running mode: multimodal
PyTorch 2.10.0+cu128
Device: cuda
Ablation mode: multimodal
GPU: NVIDIA L4
Excluding 0 classes ending with 'leaf':

Keeping 59 classes for training/testing.
Image train embeddings shape:  torch.Size([8560, 768])
Text  train embeddings shape:  torch.Size([2360, 768])
Image test  embeddings shape:  torch.Size([2140, 768])
Text  test  embeddings shape:  torch.Size([590, 768])
Unique image classes: 59
Unique text  classes: 59

Filtered embeddings:
Image train embeddings shape:  torch.Size([8560, 768])
Text  train embeddings shape:  torch.Size([2360, 768])
Image test  embeddings shape:  torch.Size([2140, 768])
Text  test  embeddings shape:  torch.Size([590, 768])
Unique filtered image classes: 59
Unique filtered text  classes: 59

Aligning embeddings...
Aligned train: img=torch.Size([8560, 768]), txt=torch.Size([8560, 768]), lbl=torch.Size([8560])
Aligned test:  img=torch.Size([2140, 768]), txt=torch.Size([2140, 768]), lbl=torch.Size([2140])

Training for 50 epochs

Testing: 100%|██████████| 67/67 [00:00<00:00, 874.10it/s]


Epoch   1/50  Loss: 1.0551  Train Acc: 91.99%  Test Acc: 82.52%
  ✓ Best model saved to ./checkpoints/best_mlp_multimodal.pt (82.52%)


Testing: 100%|██████████| 67/67 [00:00<00:00, 875.31it/s]


Epoch   2/50  Loss: 0.8409  Train Acc: 96.78%  Test Acc: 83.79%
  ✓ Best model saved to ./checkpoints/best_mlp_multimodal.pt (83.79%)


...

Epoch  49/50  Loss: 0.7702  Train Acc: 98.42%  Test Acc: 79.30%


Testing: 100%|██████████| 67/67 [00:00<00:00, 892.58it/s]


Epoch  50/50  Loss: 0.7706  Train Acc: 98.39%  Test Acc: 79.16%

Done — best Test Acc: 80.93%

Final evaluation on test set:


Evaluating: 100%|██████████| 67/67 [00:00<00:00, 902.55it/s]

  Acc: 80.93%  Precision: 80.57%  Recall: 78.78%  F1: 78.04%


---
# Generate Heatmap Data

_Run HiResCAM on trained models to produce (spatial_feat, heatmap) training pairs._

Step 1: Generate training data for the heatmap generator model.

For each training image, this script:
  1. Runs the MobileViT backbone → spatial_feat (C, H, W)
  2. Runs full HiResCAM (gradient-based) → ground-truth heatmap (1, H_out, W_out)
  3. Saves (spatial_feat, heatmap) pairs to disk

Usage:
  cd backend
  python generate_heatmap_data.py

Output:
  ./checkpoints/heatmap_training_data/
    spatial_feats.pt   — tensor of shape (N, C, H_s, W_s)
    heatmaps.pt        — tensor of shape (N, 1, 320, 320)

In [ ]:
import os
import json
import glob
import random

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import timm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# from mlp import MultimodalMLP
from mlp import FlexibleMLP
from bert import FeatureExtractorModel

# ── Configuration ─────────────────────────────────────────────────────────────

DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE        = 320
NUM_CLASSES     = 89

IMAGES_DIR      = "./data/images/plantwild"
TEXT_DATA_DIR   = "./data/text"
LABEL_MAP_PATH  = "./data/label_map.json"

IMAGE_MODEL_PATH = "./checkpoints/best_image_encoder.pt"
TEXT_MODEL_PATH  = "./checkpoints/best_text_encoder.pt"
MLP_MODEL_PATH = "./checkpoints/best_mlp_multimodal.pt"  # for FlexibleMLP
# MLP_MODEL_PATH   = "./checkpoints/best_multimodal_mlp.pt"

OUTPUT_DIR       = "./checkpoints/heatmap_training_data"
MAX_SAMPLES      = 5000  # more data helps the model learn better heatmap patterns

# ── Model setup ───────────────────────────────────────────────────────────────

class HeatmapDataGenerator:
    def __init__(self):
        print(f"Device: {DEVICE}")
        print("Loading models...")

        with open(LABEL_MAP_PATH, 'r') as f:
            self.label_map = json.load(f)

        # Image backbone (MobileViTv2)
        self.vit = timm.create_model("mobilevitv2_150", pretrained=False,
                                     num_classes=0, global_pool='')
        self.vit.load_state_dict(
            torch.load(IMAGE_MODEL_PATH, map_location=DEVICE), strict=False
        )
        self.vit.to(DEVICE).eval()

        # Hook the last three stages for multi-layer HiResCAM
        self.activations = {}
        self.gradients = {}

        def get_forward_hook(name):
            def hook(module, input, output):
                self.activations[name] = output
            return hook

        def get_backward_hook(name):
            def hook(module, grad_input, grad_output):
                self.gradients[name] = grad_output[0]
            return hook

        for i, layer_idx in enumerate([-3, -2, -1]):
            layer = self.vit.stages[layer_idx]
            layer.register_forward_hook(get_forward_hook(f"stage_{i}"))
            layer.register_full_backward_hook(get_backward_hook(f"stage_{i}"))

        # Text encoder (agriculture-BERT)
        self.tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
        bert_base = AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL_PATH)
        self.bert = FeatureExtractorModel(bert_base).to(DEVICE).eval()

        # Fusion MLP
        # self.mlp = MultimodalMLP(num_classes=NUM_CLASSES).to(DEVICE).eval()
        self.mlp = FlexibleMLP(num_classes=NUM_CLASSES, mode="multimodal").to(DEVICE).eval()
        self.mlp.load_state_dict(torch.load(MLP_MODEL_PATH, map_location=DEVICE))

        self.transform = transforms.Compose([
            transforms.Resize(int(IMG_SIZE * 1.15)),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

        print("Models loaded.\n")

    def generate_one(self, image_path, text_description, disease_name):
        """
        Returns:
            spatial_feat: (C, H, W) tensor from backbone forward pass
            heatmap:      (1, IMG_SIZE, IMG_SIZE) normalised gradient-based heatmap
        """
        raw_image = Image.open(image_path).convert('RGB')
        img_tensor = self.transform(raw_image).unsqueeze(0).to(DEVICE)
        img_tensor.requires_grad = True

        encoded_text = self.tokenizer(
            text_description, padding=True, truncation=True,
            max_length=128, return_tensors="pt"
        ).to(DEVICE)

        self.vit.zero_grad()
        self.mlp.zero_grad()

        # Forward pass
        spatial_features = self.vit(img_tensor)
        pooled_img_emb = F.adaptive_avg_pool2d(spatial_features, (1, 1)).flatten(1)

        text_emb = self.bert(
            encoded_text["input_ids"], encoded_text["attention_mask"]
        ).detach()
        logits = self.mlp(image_emb=pooled_img_emb, text_emb=text_emb)
        # logits = self.mlp(pooled_img_emb, text_emb)

        # Get target class
        target_class = self.label_map.get(disease_name)
        if target_class is None:
            target_class = logits.argmax(dim=1).item()

        # Backward pass for gradients
        score = logits[0, target_class]
        score.backward()

        # Multi-layer HiResCAM fusion
        fused_cam = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)

        for name in self.activations.keys():
            act = self.activations[name]
            grad = self.gradients[name]
            cam = torch.sum(grad * act, dim=1).squeeze()
            cam = F.relu(cam).cpu().detach().numpy()
            cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
            fused_cam += cam

        # Normalise to [0, 1]
        if fused_cam.max() > 0:
            fused_cam = (fused_cam - fused_cam.min()) / (fused_cam.max() - fused_cam.min() + 1e-8)
        else:
            fused_cam = np.zeros_like(fused_cam)

        # Get spatial_feat from forward pass (detach, move to CPU)
        spatial_feat = spatial_features.squeeze(0).detach().cpu()  # (C, H, W)

        # Heatmap as tensor (1, IMG_SIZE, IMG_SIZE)
        heatmap = torch.from_numpy(fused_cam).unsqueeze(0)  # (1, 320, 320)

        return spatial_feat, heatmap


def collect_samples(images_dir, text_data_dir):
    """Collect (image_path, text, disease_name) tuples from the dataset."""
    samples = []

    # Load all text descriptions
    all_descriptions = {}
    json_files = glob.glob(os.path.join(text_data_dir, '*.json'))
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for disease, descs in data.items():
                    if disease not in all_descriptions:
                        all_descriptions[disease] = []
                    all_descriptions[disease].extend(descs)
        except Exception:
            pass

    # Iterate class directories
    class_dirs = [d for d in glob.glob(os.path.join(images_dir, '*')) if os.path.isdir(d)]

    for class_dir in class_dirs:
        disease_name = os.path.basename(class_dir)
        if disease_name not in all_descriptions:
            continue

        valid_extensions = {".jpg", ".jpeg", ".png"}
        image_files = [f for f in glob.glob(os.path.join(class_dir, '*.*'))
                       if os.path.splitext(f)[1].lower() in valid_extensions]

        descs = all_descriptions[disease_name]
        for img_path in image_files:
            text = random.choice(descs)
            samples.append((img_path, text, disease_name))

    return samples


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    generator = HeatmapDataGenerator()

    print("Collecting samples...")
    samples = collect_samples(IMAGES_DIR, TEXT_DATA_DIR)
    random.shuffle(samples)

    if MAX_SAMPLES and len(samples) > MAX_SAMPLES:
        samples = samples[:MAX_SAMPLES]

    print(f"Generating heatmap data for {len(samples)} images...\n")

    all_spatial_feats = []
    all_heatmaps = []
    skipped = 0

    for img_path, text, disease in tqdm(samples, desc="Generating"):
        try:
            spatial_feat, heatmap = generator.generate_one(img_path, text, disease)
            all_spatial_feats.append(spatial_feat)
            all_heatmaps.append(heatmap)
        except Exception as e:
            skipped += 1
            if skipped <= 5:
                print(f"  Skipped {os.path.basename(img_path)}: {e}")

    print(f"\nDone. {len(all_spatial_feats)} pairs generated, {skipped} skipped.")

    # Stack and save
    spatial_feats_tensor = torch.stack(all_spatial_feats)  # (N, C, H_s, W_s)
    heatmaps_tensor = torch.stack(all_heatmaps)            # (N, 1, 320, 320)

    print(f"  spatial_feats shape: {spatial_feats_tensor.shape}")
    print(f"  heatmaps shape:      {heatmaps_tensor.shape}")

    torch.save(spatial_feats_tensor, os.path.join(OUTPUT_DIR, "spatial_feats.pt"))
    torch.save(heatmaps_tensor, os.path.join(OUTPUT_DIR, "heatmaps.pt"))

    print(f"\nSaved to {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()


---
# Train Heatmap Generator

_Train a lightweight CNN to predict heatmaps from spatial features._

Step 2: Train a small CNN to predict gradient-based heatmaps from spatial features.

Input:  spatial_feat (1, C, H_s, W_s) — from MobileViTv2 backbone
Output: heatmap      (1, 1, 320, 320) — mimics HiResCAM output

The model learns to approximate the gradient-based heatmap using only
forward-pass features, so it can run on-device via ONNX without needing
backpropagation.

Usage:
  cd backend
  python train_heatmap_model.py

Requires:
  ./checkpoints/heatmap_training_data/spatial_feats.pt
  ./checkpoints/heatmap_training_data/heatmaps.pt
  (generated by generate_heatmap_data.py)

Output:
  ./checkpoints/best_heatmap_generator.pt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS     = 100
LR         = 1e-3
IMG_SIZE   = 320
FEAT_SIZE  = 10   # spatial feature resolution from MobileViTv2

DATA_DIR   = "./checkpoints/heatmap_training_data"
SAVE_PATH  = "./checkpoints/best_heatmap_generator.pt"

torch.manual_seed(42)

# ── Model ─────────────────────────────────────────────────────────────────────

class HeatmapGenerator(nn.Module):
    """
    Lightweight CNN that maps spatial features → heatmap.

    Operates at the native feature resolution (10x10) then
    upsamples to output size. This avoids the impossible task of
    hallucinating 320x320 detail from 10x10 input.

    Architecture:
      Conv2d (C → 256) → ReLU → Dropout →
      Conv2d (256 → 128) → ReLU → Dropout →
      Conv2d (128 → 64) → ReLU →
      Conv2d (64 → 1) → Sigmoid →
      Bilinear upsample to (320, 320)
    """

    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.3),

            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.3),

            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, spatial_feat, upsample=True):
        """
        Args:
            spatial_feat: (B, C, H, W) from image backbone
            upsample: if True, upsample to (320, 320) for inference
        Returns:
            heatmap: (B, 1, H, W) or (B, 1, 320, 320) if upsample=True
        """
        x = self.net(spatial_feat)
        if upsample:
            x = F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode='bilinear',
                              align_corners=False)
        return x


# ── Dataset ───────────────────────────────────────────────────────────────────

class HeatmapDataset(Dataset):
    def __init__(self, spatial_feats, heatmaps, target_size=None):
        self.spatial_feats = spatial_feats
        # Downscale ground-truth heatmaps to match feature resolution
        # This makes the task feasible: predict 10x10 from 10x10, not 320x320
        if target_size and heatmaps.shape[-1] != target_size:
            self.heatmaps = F.interpolate(
                heatmaps, size=(target_size, target_size),
                mode='bilinear', align_corners=False
            )
            print(f"  Downscaled heatmaps: {heatmaps.shape} → {self.heatmaps.shape}")
        else:
            self.heatmaps = heatmaps

    def __len__(self):
        return len(self.spatial_feats)

    def __getitem__(self, idx):
        return self.spatial_feats[idx], self.heatmaps[idx]


# ── Training ──────────────────────────────────────────────────────────────────

def main():
    print(f"Device: {DEVICE}")
    print("Loading training data...")

    spatial_feats = torch.load(f"{DATA_DIR}/spatial_feats.pt", map_location="cpu")
    heatmaps = torch.load(f"{DATA_DIR}/heatmaps.pt", map_location="cpu")

    print(f"  spatial_feats: {spatial_feats.shape}")
    print(f"  heatmaps:      {heatmaps.shape}")

    in_channels = spatial_feats.shape[1]
    print(f"  in_channels:   {in_channels}")

    # Train/val split (90/10)
    # Train at feature resolution (10x10), model upsamples to 320x320 at inference
    dataset = HeatmapDataset(spatial_feats, heatmaps, target_size=FEAT_SIZE)
    n_val = max(1, int(len(dataset) * 0.1))
    n_train = len(dataset) - n_val
    train_set, val_set = random_split(dataset, [n_train, n_val],
                                       generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

    print(f"  train: {n_train}, val: {n_val}\n")

    # Model
    model = HeatmapGenerator(in_channels=in_channels).to(DEVICE)
    param_count = sum(p.numel() for p in model.parameters())
    print(f"HeatmapGenerator: {param_count:,} parameters\n")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    # Weighted MSE: penalizes missing hot spots 10x more than false positives
    # This prevents the model from learning "predict blue everywhere"
    def weighted_mse(pred, target):
        weight = 1.0 + 9.0 * target  # weight=1 for cold pixels, weight=10 for hot pixels
        return (weight * (pred - target) ** 2).mean()

    best_val_loss = float('inf')

    for epoch in range(1, EPOCHS + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0

        for feats, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}",
                                   leave=False):
            feats = feats.to(DEVICE)
            targets = targets.to(DEVICE)

            preds = model(feats, upsample=False)  # train at native 10x10
            loss = weighted_mse(preds, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * feats.size(0)

        train_loss /= n_train

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for feats, targets in val_loader:
                feats = feats.to(DEVICE)
                targets = targets.to(DEVICE)
                preds = model(feats, upsample=False)  # validate at native 10x10
                val_loss += weighted_mse(preds, targets).item() * feats.size(0)

        val_loss /= n_val
        scheduler.step(val_loss)

        lr_now = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.6f} | "
              f"val_loss: {val_loss:.6f} | lr: {lr_now:.2e}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'in_channels': in_channels,
                'val_loss': best_val_loss,
            }, SAVE_PATH)
            print(f"  → Saved best model (val_loss: {best_val_loss:.6f})")

    print(f"\nTraining complete. Best val_loss: {best_val_loss:.6f}")
    print(f"Model saved to {SAVE_PATH}")


if __name__ == "__main__":
    main()
